In [48]:
# 必要なライブラリをインポート
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib # モデルの保存に使用
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error
import os

# --- 1. データの読み込み ---
features_data_path = '../data/processed/features.parquet'
df = pd.read_parquet(features_data_path)

In [49]:
!pip install statsmodels

In [50]:
### ①ライブラリの読み込み ###
import statsmodels.api as sm #統計モデルパッケージを読み込み
from sklearn import preprocessing 

### ②説明変数・目的変数のセット ###
# 説明変数のセット
X = df[['最寄駅：距離（分）', '面積（㎡）',  '建ぺい率（％）',
       '容積率（％）','取引時点での築年数', '取引の事情等_その他事情有り',
       '取引の事情等_他の権利・負担付き', '取引の事情等_他の権利・負担付き、調停・競売等', '取引の事情等_瑕疵有りの可能性',
       '取引の事情等_調停・競売等', '取引の事情等_調停・競売等、瑕疵有りの可能性', '取引の事情等_関係者間取引',
       '取引の事情等_関係者間取引、瑕疵有りの可能性', '取引の事情等_関係者間取引、調停・競売等', 
       '改装_改装済',  '間取り_grouped_オープンフロア',
       '間取り_grouped_欠損値', '間取り_grouped_１ＤＫ', '間取り_grouped_１Ｋ',
       '間取り_grouped_１ＬＤＫ', '間取り_grouped_１Ｒ', '間取り_grouped_２ＤＫ',
       '間取り_grouped_２Ｋ', '間取り_grouped_２ＬＤＫ', '間取り_grouped_２ＬＤＫ＋Ｓ',
       '間取り_grouped_３ＤＫ', '間取り_grouped_３ＬＤＫ', '間取り_grouped_４ＤＫ',
       '間取り_grouped_４ＬＤＫ','人口密度','市区町村人口密度']]
# 目的変数のセット
Y = df['取引価格（総額）_log']

### 標準化 ###
# 説明変数の標準化（Zスコア）
X_standard =  preprocessing.scale(X) #各自入力

# 目的変数の標準化（Zスコア）
Y_standard = preprocessing.scale(Y) #各自入力

# y切片を追加設定 ※statsmodelの回帰モデルでは必須
X_const = sm.add_constant(X_standard)

### ③モデル構築 ###
# 重回帰モデルを作成
model = sm.OLS(Y_standard, X_const)  #インスタンス化（関数を使える状態にする） ※OLS=最小二乗法
results = model.fit()           #モデル構築（フィッティング）

In [51]:
X_const

array([[ 1.        , -0.19139362,  0.81935286, ..., -0.31017845,
        -1.40787124, -0.84696674],
       [ 1.        , -0.67776649, -1.04748129, ..., -0.31017845,
        -1.40787124, -0.1457427 ],
       [ 1.        ,  0.29497924,  0.25930262, ..., -0.31017845,
        -1.40787124, -0.1457427 ],
       ...,
       [ 1.        ,  3.21321642,  0.25930262, ..., -0.31017845,
        -1.17082046, -0.67803569],
       [ 1.        ,  0.94347639,  0.25930262, ..., -0.31017845,
        -1.17082046, -0.67803569],
       [ 1.        ,  0.45710353,  0.81935286, ...,  3.22395059,
        -1.17082046, -0.67803569]])

In [52]:
### ④結果の出力 ###
results.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.654
Model:                            OLS   Adj. R-squared:                  0.654
Method:                 Least Squares   F-statistic:                 3.365e+04
Date:                Wed, 08 Oct 2025   Prob (F-statistic):               0.00
Time:                        13:37:01   Log-Likelihood:            -4.8993e+05
No. Observations:              551645   AIC:                         9.799e+05
Df Residuals:                  551613   BIC:                         9.803e+05
Df Model:                          31                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       4.885e-15      0.001   6.17e-12      1.000      -0.002       0.002
x1            -0.1101      0.001   -122.987      0.000      -0.112      -0.108
x2             0.3416      0.001    279.388      0.000       0.339       0.344
x3            -0.0608      0.001    -47.925      0.000      -0.063      -0.058
x4             0.1150      0.001     87.583      0.000       0.112       0.118
x5            -0.4335      0.001   -506.154      0.000      -0.435      -0.432
x6            -0.0062      0.001     -7.822      0.000      -0.008      -0.005
x7            -0.0046      0.001     -5.805      0.000      -0.006      -0.003
x8            -0.0002      0.001     -0.293      0.769      -0.002       0.001
x9            -0.0114      0.001    -14.394      0.000      -0.013      -0.010
x10           -0.0876      0.001   -109.922      0.000      -0.089      -0.086
x11           -0.0023      0.001     -2.906      0.004      -0.004      -0.001
x12           -0.0171      0.001    -21.581      0.000      -0.019      -0.016
x13           -0.0017      0.001     -2.098      0.036      -0.003      -0.000
x14            0.0001      0.001      0.132      0.895      -0.001       0.002
x15            0.0683      0.001     83.426      0.000       0.067       0.070
x16           -0.0392      0.001    -33.760      0.000      -0.041      -0.037
x17           -0.0223      0.002    -12.372      0.000      -0.026      -0.019
x18           -0.0695      0.002    -38.644      0.000      -0.073      -0.066
x19           -0.2187      0.004    -61.790      0.000      -0.226      -0.212
x20            0.0033      0.002      1.615      0.106      -0.001       0.007
x21           -0.0843      0.001    -66.279      0.000      -0.087      -0.082
x22           -0.0228      0.002    -12.516      0.000      -0.026      -0.019
x23           -0.0181      0.001    -20.192      0.000      -0.020      -0.016
x24            0.0505      0.003     15.070      0.000       0.044       0.057
x25            0.0095      0.001      8.892      0.000       0.007       0.012
x26           -0.0251      0.001    -17.174      0.000      -0.028      -0.022
x27            0.0585      0.005     12.634      0.000       0.049       0.068
x28           -0.0140      0.001    -15.306      0.000      -0.016      -0.012
x29            0.0395      0.003     14.388      0.000       0.034       0.045
x30            0.2681      0.001    247.385      0.000       0.266       0.270
x31            0.2318      0.001    208.949      0.000       0.230       0.234
==============================================================================
Omnibus:                   292697.582   Durbin-Watson:                   1.850
Prob(Omnibus):                  0.000   Jarque-Bera (JB):         21484480.981
Skew:                          -1.721   Prob(JB):                         0.00
Ku

In [53]:
### VIF統計量を算出 ###

from statsmodels.stats.outliers_influence import variance_inflation_factor

# VIFを計算
vif = pd.DataFrame() #結果格納用のdataframeを準備
vif['VIF'] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])] #VIFを計算しdataframeに格納
vif['変数名'] = X.columns #対応する変数名を格納

# VIFの計算結果を画面出力
display(vif)

,VIF,変数名
0,3.833638,最寄駅：距離（分）
1,12.290377,面積（㎡）
2,77.844778,建ぺい率（％）
3,13.404169,容積率（％）
4,3.795103,取引時点での築年数
5,1.000219,取引の事情等_その他事情有り
6,1.000140,取引の事情等_他の権利・負担付き
7,1.000021,取引の事情等_他の権利・負担付き、調停・競売等
8,1.000191,取引の事情等_瑕疵有りの可能性
9,1.038011,取引の事情等_調停・競売等


In [41]:
# ### グラフ（stem plot）による視覚化 ###

# # Figureサイズの指定
# plt.rcParams['figure.figsize'] = 10, 5
# # VIFの傾向をグラフ化
# plt.stem(vif.index.astype(str)+'.'+vif['変数名'], vif['VIF']) #x軸はインデックス番号と変数名を結合して表示
# # x軸の目盛文字を90度回転
# plt.xticks(rotation=90)
# # y軸ラベルを表示
# plt.ylabel('VIF')

In [55]:
### VIF=10以上の説明変数を抽出 ###

# VIF>=10に絞り込み
vif_over10 = vif[ vif['VIF']>=10 ] #各自入力（参考：vif[条件式]で条件抽出）
# 画面出力
display(vif_over10)

,VIF,変数名
1,12.290377,面積（㎡）
2,77.844778,建ぺい率（％）
3,13.404169,容積率（％）
18,10.496107,間取り_grouped_１Ｋ
26,27.082769,間取り_grouped_３ＬＤＫ
30,13.375956,市区町村人口密度


In [56]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

# 予測値の計算
Y_pred_standard = results.predict(X_const)

# MAE (平均絶対誤差) の計算
mae = mean_absolute_error(Y_standard, Y_pred_standard)
print(f"MAE (平均絶対誤差): {mae}")

# RMSE (二乗平均平方根誤差) の計算
rmse = np.sqrt(mean_squared_error(Y_standard, Y_pred_standard))
print(f"RMSE (二乗平均平方根誤差): {rmse}")

MAE (平均絶対誤差): 0.4275902228804651
RMSE (二乗平均平方根誤差): 0.5881304864423209
